# 07 · Exploring the combined data

**Milestone 2, item 2.2** · see `docs/MILESTONES.md`

## What we're looking for

For the first time, every series sits in one table (`master_monthly.csv`, notebook 06).
This notebook looks at them side by side to:
- see the main patterns the app will show,
- spot anything odd before the formal analysis, and
- decide what Milestone 4 should test.

Everything here is **exploratory**: charts and simple numbers, no statistical tests yet.

**Charts**
1. Prices at the time vs. adjusted for inflation
2. All three meats on one scale (2015 = 100)
3. The whole chain on one timeline: drought → cow slaughter → herd → beef price
4. A first look at the delays between the steps
5. Who gets the beef dollar, in today's money
6. Seasonality: do prices follow a yearly pattern?
7. A single-year view, as the app will show it

It ends with a list of questions for Milestone 4.

## Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import find_dotenv

env_path = find_dotenv()
PROJECT_ROOT = Path(env_path).parent if env_path else Path.cwd().parent
df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "master_monthly.csv", parse_dates=["date"]).set_index("date")

MEATS = {"beef": "Ground beef", "pork": "Pork chops", "chicken": "Chicken breast"}
COLORS = {"beef": "#9A3B2E", "pork": "#2F6690", "chicken": "#C08A1E"}
BASE_MONTH = df["cpi_all_items"].last_valid_index()
prices = df.loc["2015":]

print(f"{len(df)} months x {df.shape[1]} columns; real prices in {BASE_MONTH:%B %Y} dollars")

## 1 · Prices at the time vs. adjusted for inflation

**What we're looking for:** how much of each meat's price rise is just general inflation.
The solid line is the price as it was each month; the dashed line is the same price in
today's dollars (D27). Where the two lines meet (at the right), they're equal by
definition. The further apart they are on the left, the more inflation there has been
since.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), sharey=False)
for ax, (meat, label) in zip(axes, MEATS.items()):
    ax.plot(prices.index, prices[f"{meat}_price"], color=COLORS[meat], linewidth=2, label="At the time")
    ax.plot(prices.index, prices[f"{meat}_price_real"], color=COLORS[meat], linewidth=1.5,
            linestyle="--", label=f"In {BASE_MONTH:%b %Y} dollars")
    ax.set_title(label, loc="left", fontweight="bold")
    ax.set_ylabel("$ per pound")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8, frameon=False)
fig.tight_layout()
plt.show()

### How to read it
Each panel is one meat. **Solid:** the price as shoppers paid it. **Dashed:** that same
price in today's dollars. If the dashed line *falls* over time, the meat got cheaper
compared with everything else; if it *rises*, it outpaced inflation.

### What it means
- **Beef (in today's dollars)** fell from about $6.10/lb in early 2015 to about $5.00
  in 2017–2019, spiked in 2020 (COVID plant shutdowns), sat around $5.50 in 2021–2023,
  then climbed to about **$6.80–7.00** in 2025–2026, now *above* its early-2015 level. Most
  of beef's real rise happened **after 2023**.
- **Pork** fell almost steadily in real terms, from about $5.70 to $4.20, apart from the
  2020 spike. It got **cheaper** compared with everything else.
- **Chicken breast** fell to about $4.00 by 2019, spiked to about $5.40 in 2022, then
  slid back to about $4.20, below its 2015 level in real terms.

## 2 · All three meats on one scale (2015 = 100)

**What we're looking for:** which meat rose fastest, on a common scale. Each price is
rebased so its 2015 average = 100 (see "Key terms: price indexes and rebasing" in
DATA_SOURCES.md). This is the app's "All meats" view (D13). The CPI (all items) is the
dashed yardstick.

In [ ]:
def rebase(s):
    return s / s.loc["2015"].mean() * 100

fig, ax = plt.subplots(figsize=(11, 5))
for meat, label in MEATS.items():
    ax.plot(prices.index, rebase(prices[f"{meat}_price"]), color=COLORS[meat], linewidth=2, label=label)
ax.plot(prices.index, rebase(prices["cpi_all_items"]), color="#666666", linestyle="--", label="All prices (CPI)")
ax.axhline(100, color="#bbbbbb", linewidth=0.8)
ax.set_ylabel("2015 average = 100")
ax.set_title("Beef, pork and chicken vs. all prices (2015 = 100)", loc="left", fontweight="bold")
ax.legend(frameon=False)
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

latest = {label: rebase(prices[f"{m}_price"]).iloc[-1] for m, label in MEATS.items()}
latest["All prices (CPI)"] = rebase(prices["cpi_all_items"]).iloc[-1]
pd.Series(latest, name=f"{BASE_MONTH:%b %Y}, 2015 = 100").round(1).to_frame()

### How to read it
All lines start near 100 in 2015; the table gives each line's latest value. Above the
dashed CPI line means faster than inflation.

### What it means
In September 2026, on the 2015 = 100 scale: **ground beef 163**, all prices 142,
**chicken breast 123**, **pork chops 109**. Beef is the only meat above the inflation
line, and it pulled away from it in 2024. Chicken rose with inflation during 2022 and then
fell back below it. Pork stayed below it the whole time.

## 3 · The whole chain on one timeline

**What we're looking for:** the D17 chain in order, top to bottom:
**drought → cow slaughter → herd → beef price.** If the chain holds, each panel's big move
should come *after* the one above it. Beef is shown in today's dollars, so the bottom
panel is the price rise beyond general inflation.

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(11, 12), sharex=True)

axes[0].fill_between(df.index, df["drought_cattle_d2_plus_pct"], color="#C08A1E", alpha=0.45)
axes[0].set_ylabel("% of area")
axes[0].set_title("1. Drought: cattle country in severe drought or worse", loc="left", fontweight="bold")

axes[1].plot(df.index, df["slaughter_beef_cows_slaughtered_12mo"] / 1e6, color="#9A3B2E", linewidth=2)
axes[1].set_ylabel("Millions per 12 months")
axes[1].set_title("2. Sell-off: beef cows slaughtered over the past 12 months", loc="left", fontweight="bold")

axes[2].step(df.index, df["herd_beef_cows_jan1"] / 1e6, where="post", color="#5B3A1E", linewidth=2)
axes[2].set_ylabel("Millions of head")
axes[2].set_title("3. Herd: beef cows on January 1", loc="left", fontweight="bold")

axes[3].plot(df.index, df["beef_price_real"], color="#9A3B2E", linewidth=2)
axes[3].set_ylabel(f"Dollars per lb ({BASE_MONTH:%b %Y} dollars)")
axes[3].set_title("4. Price: ground beef, adjusted for inflation", loc="left", fontweight="bold")

for ax in axes:
    ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

### How to read it
Four panels, one timeline (2010 to now; the price panel starts in 2015). The herd is drawn
as steps because it's counted once a year. The slaughter line is a 12-month total, so it
reacts about six months after a change in monthly slaughter.

### What it means
The chain shows up twice:
- **The earlier cycle:** drought in 2011–2013 → cow slaughter already high in 2011, then
  falling as the herd ran low → the herd bottomed out in **2014** → ranchers rebuilt
  through 2019. Beef prices were high in real terms in 2015 and *fell* through 2019 as the
  herd grew back. So the price panel starts at the tail end of the previous cycle.
- **The recent cycle:** drought from mid-2020, worst in 2022 → cow slaughter peaked in
  **2022** → the herd fell in steps from 2020 to its 2026 low → the real beef price rose
  slowly in 2021–2023 and **fastest in 2024–2026**, as the herd reached its low.

**The order matches D17:** drought, then sell-off, then a smaller herd, then higher real
prices, with each step lagging the one before.

## 4 · A first look at the delays

**What we're looking for:** roughly how long each step takes to follow the one before.

A simple way to look: shift one series later by 0, 1, 2… months (or years) and see how
closely it then moves with the other, measured by the **correlation** (−1 to 1; near 1
means they rise and fall together). The shift with the strongest correlation hints at the
delay.

**An important caution:** two series that both trend upward will look correlated even if
they're unrelated. To avoid that, we compare **year-over-year changes** (this month vs. the
same month a year earlier), not the levels. That also removes seasonal patterns. Even so,
this is only a first look; Milestone 4 tests the delays properly.

**A. Drought → cow slaughter (yearly):** each year's average drought (DSCI) vs. that
year's beef cow slaughter as a % of the January 1 herd, with slaughter shifted 0–2 years
later. We use yearly values here because monthly slaughter is noisy and seasonal, and a
cow sell-off plays out over a year rather than a month. (A first try with monthly
year-over-year changes gave no clear signal for exactly that reason.) The slaughter rate
is a level, not a change, but it doesn't trend steadily upward, so the caution above
matters less here. There are 16 years (2010–2025), so again: a hint, not a measurement.

In [ ]:
yr = pd.DataFrame({
    "drought_dsci": df["drought_cattle_dsci"].groupby(df.index.year).mean(),
    "cow_slaughter_pct_of_herd": (df["slaughter_beef_cows_slaughtered"].groupby(df.index.year).sum()
                                  / df["herd_beef_cows_jan1"].groupby(df.index.year).first() * 100),
}).loc[2010:BASE_MONTH.year - 1]   # complete years only

lag_a = pd.Series({lag: yr["drought_dsci"].corr(yr["cow_slaughter_pct_of_herd"].shift(-lag))
                   for lag in range(0, 3)}, name="correlation")

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(yr["drought_dsci"], yr["cow_slaughter_pct_of_herd"], color="#9A3B2E")
for year, row in yr.iterrows():
    ax.annotate(str(year), (row["drought_dsci"], row["cow_slaughter_pct_of_herd"]),
                textcoords="offset points", xytext=(4, 3), fontsize=8)
ax.set_xlabel("Drought in cattle country, yearly average DSCI")
ax.set_ylabel("Beef cows slaughtered, % of Jan 1 herd")
ax.set_title("Drought vs. cow slaughter, same year", loc="left", fontweight="bold")
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

print("Correlation by delay (years):")
print(lag_a.round(2).to_string())

### How to read it
Each dot is one year: how much drought cattle country had (left to right) and what share
of the beef cow herd was slaughtered (bottom to top). If drought drives the sell-off, the
dots should rise from left to right. The printed correlations show how the relationship
changes when slaughter is compared with drought one and two years earlier.

### What it means
- **Same year: correlation +0.58.** Years with more drought had more beef cows
  slaughtered: 2011, 2022 and 2023 sit at the top right. That's the expected direction.
- **Two years later: −0.39.** Two years after a drought, cow slaughter tends to be
  *lower*: fewer cows are left to sell, and ranchers start keeping them to rebuild.
- **Exceptions worth explaining in Milestone 4:**
  - **2012–2013** had heavy drought but *falling* slaughter: much of the herd had already
    been sold in 2011.
  - **2024–2025** had moderate drought but low slaughter: with record cattle prices,
    holding on to cows paid off.

  Drought alone doesn't decide the sell-off; how many cows are left, and what cattle are
  worth, matter too.

**B. Herd → beef price (yearly):** the year-over-year change in the January 1 beef cow
herd vs. the year-over-year change in the real beef price (yearly averages), with the
price shifted 0–3 years later.

There are only about ten years of overlap (prices start in 2015), so treat this as a
hint, not a measurement.

In [ ]:
yearly = pd.DataFrame({
    "herd_change_pct": df["herd_beef_cows_jan1"].groupby(df.index.year).first().pct_change(fill_method=None) * 100,
    "real_beef_change_pct": df["beef_price_real"].groupby(df.index.year).mean().pct_change(fill_method=None) * 100,
})
lag_b = pd.Series({lag: yearly["herd_change_pct"].corr(yearly["real_beef_change_pct"].shift(-lag))
                   for lag in range(0, 4)}, name="correlation")
print("Correlation by delay (years):")
print(lag_b.round(2).to_string())
yearly.loc[2015:].round(1)

### How to read it
A **negative** correlation is what the theory predicts: when the herd shrinks (negative
change), the price rises (positive change). The table below shows the yearly changes
themselves.

### What it means
- **Same year: −0.59; one year later: −0.55; two years later: −0.29; three: about 0.**
  When the herd shrinks, the real beef price tends to rise **the same year and the year
  after**, and the link fades after that. That's the expected direction.
- The yearly changes show it: the herd shrank every year from 2020 to 2026, and the real
  beef price rose in every one of those years, most in 2025 (+10%) and 2026 so far (+8%).
- With only about ten years of data this is a hint. Milestone 4 will estimate how much
  the price rises for each 1% fall in the herd, with an honest margin of error.

## 5 · Who gets the beef dollar, in today's money

**What we're looking for:** notebook 02 showed each stage's *share* of the retail price.
Here are the amounts themselves, in today's cents per pound, so we can see whose part
grew in real terms.

In [ ]:
deflator = df["cpi_all_items"].iloc[-1] / df["cpi_all_items"]
beef = pd.DataFrame({
    "Farm (rancher)": df["ers_beef_net_farm"] * deflator,
    "Packer (farm to wholesale)": df["ers_beef_spread_farm_to_wholesale"] * deflator,
    "Store (wholesale to retail)": df["ers_beef_spread_wholesale_to_retail"] * deflator,
}).loc["2015":]

fig, ax = plt.subplots(figsize=(11, 4.8))
ax.stackplot(beef.index, beef.T.values, labels=beef.columns, colors=["#5B3A1E", "#B07A4A", "#E3C9A8"])
ax.set_ylabel(f"Cents per lb ({BASE_MONTH:%b %Y} cents)")
ax.set_title("Beef retail value split by stage, adjusted for inflation", loc="left", fontweight="bold")
ax.legend(loc="upper left", fontsize=8)
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

beef.groupby(beef.index.year).mean().loc[[2015, 2019, 2022, 2025]].round(0)

### How to read it
The top of the stack is the full retail value of beef (all cuts, in today's cents per
pound); the three bands are how it splits. The table gives yearly averages for four
years.

### What it means
In today's cents per pound of retail beef:

| | 2015 | 2019 | 2022 | 2025 |
|---|---|---|---|---|
| Farm (rancher) | 461 | 339 | 354 | **524** |
| Packer | 55 | 111 | 105 | **46** |
| Store | 378 | 346 | 415 | 410 |

- Since 2019, the **rancher's part rose by about 185 cents** per pound in real terms, the
  store's by about 64, and the **packer's fell by about 65**. Most of the real increase
  in beef's retail value went to the farm level, where the cattle are scarce.
- The store's part rose too (2022 onward), but much less. ERS spreads include all costs
  (labor, energy, transport), so a larger spread isn't necessarily larger profit.

## 6 · Seasonality

**What we're looking for:** whether prices follow a regular yearly pattern, for example
higher in grilling season. If they do, the app's single-year view (D11) should mention
it, so a summer rise isn't mistaken for an event.

For each month we compare the price with the average of the surrounding 12 months (a
centered 12-month average), then average that difference for each calendar month across
2015–2026. A bar of +2% means that month is typically 2% above the yearly trend.

In [ ]:
season = {}
for meat, label in MEATS.items():
    s = prices[f"{meat}_price"]
    trend = s.rolling(12, center=True).mean()
    dev = (s / trend - 1) * 100
    season[label] = dev.groupby(dev.index.month).mean()
season = pd.DataFrame(season)
season.index = pd.to_datetime(season.index, format="%m").strftime("%b")

ax = season.plot.bar(figsize=(11, 4), color=[COLORS[m] for m in MEATS], width=0.8)
ax.axhline(0, color="#888888", linewidth=0.8)
ax.set_ylabel("% above / below the 12-month trend")
ax.set_title("Typical seasonal pattern, 2015-2026", loc="left", fontweight="bold")
ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

season.round(1).T

### How to read it
Each group of bars is a calendar month; each bar is how far that meat's price typically
sits above (positive) or below (negative) its yearly trend in that month.

### What it means
- There **is** a small seasonal pattern: **beef** is typically about 2% above trend in
  June (grilling season) and 1–1.5% below in January–March; **pork** follows a similar
  summer high (+1.8%) and January low (−2.8%); **chicken's** pattern is weaker.
- These swings are **small** (a few percent) compared with the 20–60% multi-year changes
  this project is about, so they don't change any conclusion. But the app's single-year
  view should mention them, so a June rise isn't mistaken for an event.

## 7 · A single-year view (prototype)

**What we're looking for:** a first draft of what the app shows when the user picks one
year (D11): the year month by month, the year before for comparison, and the change.
Change `YEAR` and re-run the cell to look at another year.

In [ ]:
YEAR = 2021
MEAT = "beef"

this_year = prices.loc[str(YEAR), f"{MEAT}_price"]
last_year = prices.loc[str(YEAR - 1), f"{MEAT}_price"]
months = this_year.index.strftime("%b")

fig, ax = plt.subplots(figsize=(10, 4.2))
ax.plot(months, this_year.values, color=COLORS[MEAT], linewidth=2.4, marker="o", label=str(YEAR))
ax.plot(months[:len(last_year)], last_year.values[:len(months)], color="#999999", linewidth=1.5,
        marker="o", linestyle="--", label=str(YEAR - 1))
ax.set_ylabel("$ per pound")
ax.set_title(f"{MEATS[MEAT]}, {YEAR} month by month", loc="left", fontweight="bold")
ax.legend(frameon=False)
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

change = (this_year.mean() / last_year.mean() - 1) * 100
print(f"{YEAR} average ${this_year.mean():.2f}/lb vs {YEAR - 1} ${last_year.mean():.2f}/lb: {change:+.1f}%")
print(f"Lowest {months[this_year.argmin()]} (${this_year.min():.2f}), highest {months[this_year.argmax()]} (${this_year.max():.2f})")

### How to read it
The solid line is the chosen year, the dashed gray line the year before. The two printed
lines give the yearly averages, the change, and the lowest and highest months.

### What it means
For **2021**, ground beef averaged **$4.33/lb, up 5.2% on 2020**. It rose steadily all
year, from $3.96 in January to $4.72 in October. The 2020 line shows the COVID spike
(May–June 2020) and how quickly it reversed. The jump from May to June 2021 lines up with
the JBS cyberattack, one of the events Milestone 3 will test.

This layout works for the app: one year, the year before as a reference, and a one-line
summary.

## Questions for Milestone 4

From what the charts show, these are the questions worth testing properly:

1. **Beef vs. inflation:** how large is beef's real price rise since 2015 and 2019, and how
   confident can we be that it's different from pork and chicken? (items 4.1, 4.5)
2. **Drought → sell-off:** how much does drought raise cow slaughter, once we account for
   how many cows are left and what cattle are worth? Can that explain 2012–2013 and
   2024–2025? (item 4.4)
3. **Herd → price:** how much does the real beef price rise for each 1% fall in the
   herd, and over what delay (the charts suggest 0–1 year)? (item 4.4)
4. **Events:** did the price really shift at the 2020 plant shutdowns, the June 2021 JBS
   cyberattack, the 2022 chicken spike and beef's 2024 break-away? (items 4.3 and
   Milestone 3)
5. **Who gets the dollar:** is the store's larger real spread since 2022 a lasting change
   or a return to 2015 levels? (item 4.2)
6. **The 2026 drought:** it's as widespread as 2022's, but cow slaughter keeps falling.
   Worth watching, and worth a note in the app.

## What's next and why

**Milestone 3: the event timeline.** Several of these questions depend on *when* things
happened: the plant shutdowns, the JBS cyberattack, import suspensions, screwworm, and the
popular explanations we agreed to test (the 2022 plant fires and the Colorado culls). Item
3.1 researches each event and records it with its date and source in
`data/processed/events.csv`, so the charts and tests can use it.